# 04 - Train Baseline

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Cari root repo: di laptop = folder di atas notebooks/, di Kaggle = dataset mlc-src
if os.path.exists("/kaggle/input"):
    hits = list(Path("/kaggle/input").rglob("src/data.py"))
    if hits:
        ROOT = hits[0].parent.parent
    else:  # dataset masih berupa zip
        ROOT = Path("/kaggle/working/_repo")
        for z in Path("/kaggle/input").rglob("*.zip"):
            if z.stem in ("src", "configs", "annotations"):
                zipfile.ZipFile(z).extractall(ROOT / z.stem)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import ON_KAGGLE, get_path, load_config
from src.seed import seed_everything

cfg = load_config()
seed_everything(cfg["SEED"])
print("Kaggle:", ON_KAGGLE, "| data:", get_path(cfg, "raw"))

## Fitur jendela dari embedding (hasil notebook 03)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from src.data import get_path
from src.index import muat_index
from src.metrics import cari_threshold, ringkas_teks, skor, skor_per_fold
from src.models import fitur_dari_index, muat_test

METRIK = cfg["METRIC"]            # metrik lomba; jangan pakai default "f1" di src/metrics.py
SEQ_LEN = cfg["SEQ_LEN"]
dir_fitur = get_path(cfg, "features", read=True)
print("fitur dari:", dir_fitur, "| metrik:", METRIK, "| seq_len:", SEQ_LEN)

idx = muat_index(cfg)
idx = idx[idx["dipakai"]].reset_index(drop=True)     # buang potongan tepat setengah (label tidak jelas)
X, idx, lap = fitur_dari_index(idx, dir_fitur, seq_len=SEQ_LEN)   # [n, 3 x 768]: mean, max, std
idx = idx.reset_index(drop=True)                      # baris idx sejajar dengan baris X

y = idx["label"].to_numpy(int)
fold = idx["fold"].to_numpy(int)
val_mask = idx["grid30"].to_numpy(bool)               # validasi = potongan tanpa geser (bentuk test)

print("jendela dipakai:", len(idx), "| dibuang karena melewati panjang embedding:", lap["dibuang_lewat"],
      "| dibuang karena embedding tidak ada:", lap["dibuang_tanpa_embedding"])
if lap["video_hilang"]:
    print("video tanpa embedding (maks 5):", lap["video_hilang"][:5])
print("bentuk X:", X.shape, "| rasio label 1:", round(float(y.mean()), 4),
      "| jendela validasi:", int(val_mask.sum()))

## 5 fold CV + pilih C lewat validasi

In [ ]:
C_GRID = [0.01, 0.1, 1]
daftar_fold = sorted(set(fold.tolist()))


def latih(Xtr, ytr, C):
    """StandardScaler di-fit hanya pada train fold, tidak pernah pada validasi/test."""
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(C=C, class_weight="balanced", max_iter=1000)
    m.fit(sc.transform(Xtr), ytr)
    return sc, m


hasil = {}
for C in C_GRID:
    prob = np.full(len(idx), np.nan)
    model_fold = {}
    for f in daftar_fold:
        tr = fold != f                      # train: semua potongan fold lain (dipakai == True)
        va = (fold == f) & val_mask         # validasi: potongan grid30 fold ini
        sc, m = latih(X[tr], y[tr], C)
        prob[va] = m.predict_proba(sc.transform(X[va]))[:, 1]
        model_fold[f] = (sc, m)
    oof = pd.DataFrame({"fold": fold[val_mask], "label": y[val_mask], "prob": prob[val_mask]})
    thr, sk = cari_threshold(oof["label"], oof["prob"], metrik=METRIK)   # threshold dari OOF train saja
    hasil[C] = {"oof": oof, "thr": thr, "skor": sk, "model": model_fold}
    print("C=%-5s %s OOF = %.4f (thr=%.2f)" % (C, METRIK, sk, thr))

C_terbaik = max(C_GRID, key=lambda c: hasil[c]["skor"])
print("C terpilih:", C_terbaik)

## Skor OOF total dan per fold

In [ ]:
pilih = hasil[C_terbaik]
oof, thr = pilih["oof"], pilih["thr"]

tabel, ringkasan = skor_per_fold(oof, metrik=METRIK, threshold=thr)
print("%s OOF total : %.4f" % (METRIK, skor(oof["label"], oof["prob"], METRIK, thr)))
print(ringkas_teks(ringkasan))
print(tabel.to_string(index=False))

oof_simpan = oof.copy()
oof_simpan.insert(0, "video", idx.loc[val_mask, "video"].to_numpy())
oof_simpan.insert(1, "mulai_frame", idx.loc[val_mask, "mulai_frame"].to_numpy())
p_oof = get_path(cfg, "oof") / "oof_04.csv"
p_oof.parent.mkdir(parents=True, exist_ok=True)
oof_simpan.to_csv(p_oof, index=False)
print("OOF tersimpan:", p_oof)

## Prediksi test + submission

In [ ]:
# Tiap sequence test diprediksi sendiri dari 30 frame-nya; tidak ada statistik dari data test
# dan tidak ada hubungan antar-sequence (nomor id test tidak dipakai sebagai fitur).
Xte, ids = muat_test(dir_fitur, seq_len=SEQ_LEN)
prob_te = np.mean([m.predict_proba(sc.transform(Xte))[:, 1] for sc, m in pilih["model"].values()], axis=0)
pred = pd.DataFrame({"id": ids, "label": (prob_te >= thr).astype(int)})

contoh = pd.read_csv(get_path(cfg, "raw") / "sample_submission.csv")


def kunci(s):
    """Cocokkan id lewat teks, supaya id angka maupun teks sama-sama jalan.
    Kolom id yang ditulis tetap milik sample_submission (urutan dan dtype aslinya)."""
    return s.astype(str).str.strip()


sub = contoh[["id"]].copy()
sub["_k"] = kunci(sub["id"])
sub = sub.merge(pred.assign(_k=kunci(pred["id"]))[["_k", "label"]], on="_k", how="left").drop(columns="_k")

assert sub["label"].notna().all(), (
    "ada id di sample_submission yang tidak punya prediksi | contoh id sample=%r vs emb_test=%r"
    % (contoh["id"].iloc[0], ids[0]))
sub["label"] = sub["label"].astype(int)
assert len(sub) == 1515, "jumlah baris %d, seharusnya 1515" % len(sub)
assert sub["id"].tolist() == contoh["id"].tolist(), "urutan id tidak sama dengan sample_submission"
assert set(sub["label"].unique()) <= {0, 1}, "label bukan 0/1"
assert sub["label"].dtype.kind == "i", "label bukan integer"

out = get_path(cfg, "submissions") / "submission_04.csv"
out.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out, index=False)
print("Tersimpan:", out, "|", len(sub), "baris | rasio label 1:", round(float(sub["label"].mean()), 4))
print(sub.head(3).to_string(index=False))


## Catat ke logs/experiments.csv

In [ ]:
import datetime

from src.data import ON_KAGGLE

KOLOM_LOG = ["date", "model", "cv_score", "lb_score", "notes"]
baris = {
    "date": datetime.date.today().isoformat(),
    "model": "04_logreg_mean_max_std",
    "cv_score": round(float(ringkasan["oof"]), 4),
    "lb_score": "",
    "notes": "C=%s thr=%.2f %s per-fold=%.4f+-%.4f n_train=%d n_oof=%d dibuang_lewat=%d" % (
        C_terbaik, thr, METRIK, ringkasan["mean"], ringkasan["std"],
        len(idx), len(oof), lap["dibuang_lewat"]),
}

p_log = get_path(cfg, "logs")
if ON_KAGGLE:      # folder repo di Kaggle hanya-baca
    p_log = Path("/kaggle/working") / cfg["paths"]["logs"]
log = pd.read_csv(p_log) if p_log.exists() else pd.DataFrame(columns=KOLOM_LOG)
log = pd.concat([log, pd.DataFrame([baris])[KOLOM_LOG]], ignore_index=True)
p_log.parent.mkdir(parents=True, exist_ok=True)
log.to_csv(p_log, index=False)
print("Dicatat di", p_log)
print(log.tail(3).to_string(index=False))